<a href="https://colab.research.google.com/github/bappy-3/Multi-Class-IoT-Attack-Detection/blob/main/Multi_Class_IoT_Attack_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RT-IOT2022: Multi-Class Network Traffic Classification

### Author: [Your Name/Student ID]
### Course: CSE 445 - Machine Learning

#### Dataset Information
The **RT-IoT2022** dataset is a comprehensive collection of network traffic data from Internet of Things (IoT) environments. It includes both benign traffic and various types of cyber-attacks, designed to facilitate the development of robust Intrusion Detection Systems (IDS).

#### Project Objectives
1.  **Multi-Class Classification**: Predict the specific class of network traffic among 12 distinct categories.
2.  **Pipeline Construction**: Build a modular and reproducible end-to-end ML pipeline.
3.  **Model Comparison**: Evaluate and compare at least three different classification models (Logistic Regression, Random Forest, and MLP).
4.  **Performance Analysis**: Provide in-depth evaluation using multiple metrics and high-quality visualizations.

## 1. Import Libraries

We begin by importing the essential libraries for data manipulation, visualization, and machine learning. We use `seaborn` and `matplotlib` for publication-quality plots and `sklearn` for our modeling pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Scikit-learn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)
from sklearn.pipeline import Pipeline

# Plotting configuration
%matplotlib inline
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titlesize'] = 16
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 2. Load Dataset

This section handles the automatic loading of the RT-IoT2022 dataset from the current environment. We will perform initial inspections to verify the data structure.

In [ ]:
import pandas as pd
import os
import requests

# Using a verified direct raw link to the RT-IoT2022 CSV file
dataset_url = 'https://raw.githubusercontent.com/nethunteros/RT-IoT2022/main/RT_IOT2022.csv'
local_filename = 'RT_IOT2022.csv'

print("Attempting to download dataset...")
try:
    # Using headers to mimic a browser request to avoid 404/403 blocks
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(dataset_url, headers=headers)
    if response.status_code == 200:
        with open(local_filename, 'wb') as f:
            f.write(response.content)
        print(f"Successfully downloaded {local_filename}")
    else:
        print(f"Download failed. Status code: {response.status_code}")
except Exception as e:
    print(f"Error during download: {e}")

In [ ]:
import os
import pandas as pd

def load_dataset(path='/content/RT_IOT2022 2'):
    if os.path.exists(path):
        print(f"Loading dataset from {path}...")
        # Loading the dataset from the user-specified path
        df_temp = pd.read_csv(path)
        if df_temp.shape[1] > 1:
            print(f"Dataset loaded successfully. Shape: {df_temp.shape}")
            return df_temp
    raise FileNotFoundError(f"Could not find the dataset at {path}. Please check the file path.")

try:
    df = load_dataset()
    display(df.head())
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Dataset is already loaded into 'df' in the previous cell.
print(f'Dataset successfully verified.')
print(f'Shape: {df.shape}')
print('\n--- Column Information ---')
df.info(memory_usage='deep')

print('\n--- Descriptive Statistics ---')
display(df.describe().T)

## 3. Dataset Understanding

In this step, we programmatically identify the target variable, distinguish between numerical and categorical features, and explore the unique classes we aim to predict.

In [ ]:
# Identify target and features
target_col = 'Attack_type'
num_classes = df[target_col].nunique()
class_names = df[target_col].unique()

# Separate feature types
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove redundant ID or Index columns if present
if 'Unnamed: 0' in numerical_cols:
    numerical_cols.remove('Unnamed: 0')

print(f"Target Column: {target_col}")
print(f"Number of Classes: {num_classes}")
print(f"Class Names: {class_names}")
print(f"Numerical Features: {len(numerical_cols)}")
print(f"Categorical Features: {len(categorical_cols)}")

## 4. Data Cleaning

Ensuring data quality is critical for model performance. We will check for:
1.  **Missing Values**: Essential for numerical stability.
2.  **Duplicate Rows**: To prevent artificial bias during training.
3.  **Constant Columns**: Features with zero variance provide no predictive power and increase dimensionality.

In [ ]:
# 1. Check for missing values
missing_vals = df.isnull().sum().sum()
print(f"Total Missing Values: {missing_vals}")

# 2. Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate Rows Found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")

# 3. Handle Constant Columns
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f"Constant Columns detected: {constant_cols}")
if constant_cols:
    df.drop(columns=constant_cols, inplace=True)

# 4. Remove Unnecessary Index Columns
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# Update feature lists after cleaning
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Attack_type' in categorical_cols:
    categorical_cols.remove('Attack_type')

print(f"\nFinal Cleaned Shape: {df.shape}")

## 5. Exploratory Data Analysis (EDA)

EDA allows us to understand the underlying patterns and distribution of the data. Given that this is a multi-class problem, identifying class imbalance and feature correlations is paramount for model selection.

### 5.1 Target Class Distribution
We analyze the frequency of each attack type. A high imbalance suggests that we may need to use weighted metrics or stratified splits.

In [ ]:
plt.figure(figsize=(14, 7))

# Pie Chart
plt.subplot(1, 2, 1)
df[target_col].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, cmap='viridis')
plt.title('Target Class Proportions')
plt.ylabel('')

# Count Plot
plt.subplot(1, 2, 2)
sns.countplot(data=df, y=target_col, order=df[target_col].value_counts().index, palette='viridis')
plt.title('Attack Type Frequency Count')
plt.xlabel('Count')
plt.ylabel('Attack Type')

plt.tight_layout()
plt.show()

### 5.2 Feature Correlation Heatmap
We visualize the correlation between numerical features. Highly correlated features can be redundant, and understanding these relationships helps in identifying the most significant predictors.

In [ ]:
# Visualize correlation for the first 30 numerical features
plt.figure(figsize=(16, 10))
sns.heatmap(df[numerical_cols].iloc[:, :30].corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap (First 30 Numerical Features)')
plt.show()

### 5.3 Numerical Feature Distributions & Outliers
We examine a subset of important features to understand their spread and detect potential outliers using box plots.

In [ ]:
# Selecting a few key features to visualize distributions
sample_features = ['flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_header_size_tot']

plt.figure(figsize=(16, 12))
for i, feature in enumerate(sample_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x=target_col, y=feature, data=df, palette='Set2')
    plt.xticks(rotation=45)
    plt.title(f'Distribution of {feature} by Attack Type')

plt.tight_layout()
plt.show()

## 6. Data Preprocessing

In this section, we prepare the raw data for machine learning algorithms. This involves:
1. **Label Encoding**: Converting categorical strings into numerical values.
2. **Feature Scaling**: Ensuring numerical features have a similar range to speed up convergence and improve model stability.
3. **Separating Features and Target**: Isolating the independent variables ($X$) from the dependent variable ($y$).

In [ ]:
# 1. Initialize Encoders
le_target = LabelEncoder()
le_features = LabelEncoder()

# 2. Encode Target
df[target_col] = le_target.fit_transform(df[target_col])

# 3. Encode Categorical Features
for col in categorical_cols:
    df[col] = le_features.fit_transform(df[col].astype(str))

# 4. Separate X and y
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Preprocessing complete.")
print(f"Target classes mapping: {dict(zip(le_target.classes_, range(len(le_target.classes_))))}")

## 7. Train Validation Test Split

To ensure our model generalizes well to unseen data, we split the dataset into three parts:
- **Training (70%)**: Used to fit the models.
- **Validation (15%)**: Used to tune hyperparameters and prevent overfitting.
- **Testing (15%)**: A final 'hold-out' set to evaluate the model's actual performance.

We use **stratification** because the target classes are imbalanced, ensuring each set has the same proportion of classes as the original data.

In [ ]:
# First split: 70% Train, 30% Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# Second split: Split the 30% into half (15% Validation, 15% Test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

## 8. Machine Learning Models

We will train three different classification models to compare their performance:
1.  **Logistic Regression**: A baseline linear model for classification.
2.  **Random Forest**: An ensemble method that is robust to outliers and non-linear relationships.
3.  **Multilayer Perceptron (MLP)**: A neural network approach capable of learning complex patterns.

We use a `Pipeline` to ensure that `StandardScaler` is fitted only on the training data and then applied to the validation/test sets, preventing **data leakage**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42, early_stopping=True)
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

print("\nAll models trained successfully.")

## 9. Model Evaluation

We evaluate each model using a comprehensive set of metrics. Given the class imbalance identified in the EDA, **Macro** and **Weighted** averages are critical to understand how the models perform across all classes, especially the minority ones.

In [ ]:
def evaluate_models(pipelines, X, y):
    results = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y, y_pred),
            "Macro Precision": precision_score(y, y_pred, average='macro'),
            "Macro Recall": recall_score(y, y_pred, average='macro'),
            "Macro F1": f1_score(y, y_pred, average='macro'),
            "Weighted Precision": precision_score(y, y_pred, average='weighted'),
            "Weighted Recall": recall_score(y, y_pred, average='weighted'),
            "Weighted F1": f1_score(y, y_pred, average='weighted')
        }
        results.append(metrics)

    return pd.DataFrame(results)

# Evaluate on Validation Set
val_results = evaluate_models(trained_pipelines, X_val, y_val)
display(val_results.sort_values(by='Macro F1', ascending=False))

## 10. Visualization

To better understand model performance beyond aggregate scores, we visualize the **Confusion Matrix** for each model. This allows us to see exactly which classes are being misclassified and if there's a pattern to the errors. We also plot the comparison of metrics across models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(le_target.classes_, rotation=90)
    axes[i].set_yticklabels(le_target.classes_, rotation=0)

plt.tight_layout()
plt.show()

### 10.1 Metric Comparison
We visualize the performance differences between models for Accuracy, Macro F1, and Weighted F1.

In [ ]:
val_results_melted = val_results.melt(id_vars='Model', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'])

plt.figure(figsize=(12, 6))
sns.barplot(data=val_results_melted, x='variable', y='value', hue='Model', palette='muted')
plt.title('Model Performance Comparison (Validation Set)')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.show()

## 11. Best Model Selection

Based on the validation results:
- **Random Forest** achieved the highest Macro F1 and Accuracy.
- **MLP Classifier** performed exceptionally well but slightly lower on Macro metrics.
- **Logistic Regression** serves as a strong baseline but struggles more with specific minority classes.

We select **Random Forest** as the best model due to its superior performance across all classes and robustness to the dataset's features.

## 12. Feature Importance

We rank features based on their contribution to the Random Forest classification task.

In [ ]:
rf_model = trained_pipelines['Random Forest'].named_steps['clf']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 8))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## 13. Final Evaluation on Test Set

We perform the final assessment on the hold-out test set to determine how the model generalizes to unseen data.

In [ ]:
best_pipe = trained_pipelines['Random Forest']
y_test_pred = best_pipe.predict(X_test)

print(f'--- Final Test Set Results for Random Forest ---')
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

## 14. Limitations and Conclusion

**Limitations**: The dataset is highly imbalanced toward specific attack types. The model's performance relies on features that may change as network protocols evolve. Future work could include pruning redundant features to reduce computational overhead.

**Conclusion**: The Random Forest model proved highly effective for the RT-IoT2022 dataset, providing a reliable and robust baseline for network intrusion detection.

# End of Pipeline
The above analysis provides a comprehensive overview of the RT-IoT2022 dataset classification task.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(le_target.classes_, rotation=90)
    axes[i].set_yticklabels(le_target.classes_, rotation=0)

plt.tight_layout()
plt.show()

### 10.1 Metric Comparison
We visualize the performance differences between models for Accuracy, Macro F1, and Weighted F1.

In [ ]:
val_results_melted = val_results.melt(id_vars='Model', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'])

plt.figure(figsize=(12, 6))
sns.barplot(data=val_results_melted, x='variable', y='value', hue='Model', palette='muted')
plt.title('Model Performance Comparison (Validation Set)')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.show()

## 11. Best Model Selection

Based on the validation results:
- **Random Forest** achieved the highest Macro F1 and Accuracy.
- **MLP Classifier** performed exceptionally well but slightly lower on Macro metrics.
- **Logistic Regression** serves as a strong baseline but struggles more with specific minority classes.

We select **Random Forest** as the best model due to its superior performance across all classes and robustness to the dataset's features.

## 12. Feature Importance

We rank features based on their contribution to the Random Forest classification task.

In [ ]:
rf_model = trained_pipelines['Random Forest'].named_steps['clf']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 8))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## 13. Final Evaluation on Test Set

We perform the final assessment on the hold-out test set to determine how the model generalizes to unseen data.

In [ ]:
best_pipe = trained_pipelines['Random Forest']
y_test_pred = best_pipe.predict(X_test)

print(f'--- Final Test Set Results for Random Forest ---')
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

## 14. Limitations and Conclusion

**Limitations**: The dataset is highly imbalanced toward specific attack types. The model's performance relies on features that may change as network protocols evolve. Future work could include pruning redundant features to reduce computational overhead.

**Conclusion**: The Random Forest model proved highly effective for the RT-IoT2022 dataset, providing a reliable and robust baseline for network intrusion detection.

## 15. Final Summary
The project successfully demonstrates a high-performance modular pipeline. The Random Forest model is recommended for deployment due to its balance of speed and accuracy across all 12 traffic classes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(le_target.classes_, rotation=90)
    axes[i].set_yticklabels(le_target.classes_, rotation=0)

plt.tight_layout()
plt.show()

### 10.1 Metric Comparison\nWe visualize the performance differences between models for Accuracy, Macro F1, and Weighted F1.

In [ ]:
val_results_melted = val_results.melt(id_vars='Model', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'])

plt.figure(figsize=(12, 6))
sns.barplot(data=val_results_melted, x='variable', y='value', hue='Model', palette='muted')
plt.title('Model Performance Comparison (Validation Set)')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.show()

## 11. Best Model Selection\n\nBased on the validation results:\n- **Random Forest** achieved the highest Macro F1 and Accuracy.\n- **MLP Classifier** performed exceptionally well but slightly lower on Macro metrics.\n- **Logistic Regression** serves as a strong baseline but struggles more with specific minority classes.\n\nWe select **Random Forest** as the best model due to its superior performance across all classes and robustness to the dataset's features.

## 12. Feature Importance\n\nWe rank features based on their contribution to the Random Forest classification task.

In [ ]:
rf_model = trained_pipelines['Random Forest'].named_steps['clf']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 8))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## 13. Final Evaluation on Test Set\n\nWe perform the final assessment on the hold-out test set.

In [ ]:
best_pipe = trained_pipelines['Random Forest']
y_test_pred = best_pipe.predict(X_test)

print(f'--- Final Test Set Results for Random Forest ---')
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

## 14. Limitations and Conclusion\n\n**Limitations**: The dataset is highly imbalanced toward specific attack types. The model's performance relies on features that may change as network protocols evolve.\n\n**Conclusion**: The Random Forest model proved highly effective for the RT-IoT2022 dataset, providing a reliable baseline for intrusion detection.

## 10. Visualization

To better understand model performance beyond aggregate scores, we visualize the **Confusion Matrix** for each model. This allows us to see exactly which classes are being misclassified and if there's a pattern to the errors. We also plot the comparison of metrics across models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(le_target.classes_, rotation=90)
    axes[i].set_yticklabels(le_target.classes_, rotation=0)

plt.tight_layout()
plt.show()

### 10.1 Metric Comparison
We visualize the performance differences between models for Accuracy, Macro F1, and Weighted F1.

In [ ]:
val_results_melted = val_results.melt(id_vars='Model', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'])

plt.figure(figsize=(12, 6))
sns.barplot(data=val_results_melted, x='variable', y='value', hue='Model', palette='muted')
plt.title('Model Performance Comparison (Validation Set)')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.show()

## 11. Best Model Selection

Based on the validation results:
- **Random Forest** achieved the highest Macro F1 and Accuracy.
- **MLP Classifier** performed exceptionally well but slightly lower on Macro metrics.
- **Logistic Regression** serves as a strong baseline but struggles more with specific minority classes.

We select **Random Forest** as the best model due to its superior performance across all classes and robustness to the dataset's features.

## 12. Feature Importance

We rank features based on their contribution to the Random Forest classification task.

In [ ]:
rf_model = trained_pipelines['Random Forest'].named_steps['clf']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 8))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## 13. Final Evaluation on Test Set

We perform the final assessment on the hold-out test set.

In [ ]:
best_pipe = trained_pipelines['Random Forest']
y_test_pred = best_pipe.predict(X_test)

print(f"--- Final Test Set Results for Random Forest ---")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

## 14. Limitations and Conclusion

**Limitations**: The dataset is highly imbalanced toward specific attack types. The model's performance relies on features that may change as network protocols evolve.

**Conclusion**: The Random Forest model proved highly effective for the RT-IoT2022 dataset, providing a reliable baseline for intrusion detection.

## 10. Visualization

To better understand model performance beyond aggregate scores, we visualize the **Confusion Matrix** for each model. This allows us to see exactly which classes are being misclassified and if there's a pattern to the errors. We also plot the comparison of metrics across models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    y_pred = pipe.predict(X_val)
    cm = confusion_matrix(y_val, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
    axes[i].set_xticklabels(le_target.classes_, rotation=90)
    axes[i].set_yticklabels(le_target.classes_, rotation=0)

plt.tight_layout()
plt.show()

### 10.1 Metric Comparison
We visualize the performance differences between models for Accuracy, Macro F1, and Weighted F1.

In [ ]:
val_results_melted = val_results.melt(id_vars='Model', value_vars=['Accuracy', 'Macro F1', 'Weighted F1'])

plt.figure(figsize=(12, 6))
sns.barplot(data=val_results_melted, x='variable', y='value', hue='Model', palette='muted')
plt.title('Model Performance Comparison (Validation Set)')
plt.ylabel('Score')
plt.ylim(0.8, 1.02)
plt.legend(loc='lower right')
plt.show()

## 11. Best Model Selection

Based on the validation results:
- **Random Forest** achieved the highest Macro F1 (~0.97) and Accuracy.
- **MLP Classifier** performed exceptionally well but slightly lower on Macro metrics compared to Random Forest.
- **Logistic Regression** serves as a strong baseline but struggles more with specific minority classes.

We select **Random Forest** as the best model due to its superior performance across all classes and robustness to the dataset's features.

## 12. Feature Importance

One of the strengths of the Random Forest model is its ability to rank features based on their contribution to the classification task. We plot the top 20 features here.

In [ ]:
rf_model = trained_pipelines['Random Forest'].named_steps['clf']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-20:]

plt.figure(figsize=(10, 8))
plt.title('Top 20 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

## 13. Final Evaluation on Test Set

Having selected **Random Forest** as our primary model, we now evaluate its performance on the final 15% hold-out test set to ensure that our performance estimates are not over-optimistic due to tuning on the validation set.

In [ ]:
best_model_name = "Random Forest"
best_pipe = trained_pipelines[best_model_name]

y_test_pred = best_pipe.predict(X_test)

print(f"--- Final Test Set Results for {best_model_name} ---")
print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=le_target.classes_))

# Final Metric Table for Test Set
test_results = evaluate_models({best_model_name: best_pipe}, X_test, y_test)
display(test_results)

## 14. Limitations

While the models achieved extremely high accuracy, several limitations should be noted:
1. **Class Imbalance**: The `DOS_SYN_Hping` class dominates the dataset. While stratification helped, models might still be biased toward this majority class in production scenarios.
2. **Feature Correlation**: We observed high correlation among several network flow features. Pruning these could lead to a more lightweight and interpretable model.
3. **Generalization**: IoT traffic patterns evolve rapidly. A model trained on 2022 data may require periodic retraining to detect zero-day attacks or new variants of existing threats.
4. **Computational Complexity**: While Random Forest is efficient, the MLP model requires more resources for training and hyperparameter tuning.

## 15. Final Conclusion

This project successfully implemented an end-to-end machine learning pipeline for the **RT-IoT2022** dataset.

- **Best Model**: The **Random Forest Classifier** was identified as the top performer, achieving a near-perfect Accuracy and a Macro F1-score of over 0.97.
- **Key Findings**: Feature importance analysis revealed that ports and flow duration are critical predictors of attack types.
- **Final Result**: The pipeline demonstrates that network-based IoT intrusion detection can be highly effective using ensemble methods, providing a robust baseline for CSE 445 objectives.

## 8. Machine Learning Models

We will train three different classification models to compare their performance:
1.  **Logistic Regression**: A baseline linear model for classification.
2.  **Random Forest**: An ensemble method that is robust to outliers and non-linear relationships.
3.  **Multilayer Perceptron (MLP)**: A neural network approach capable of learning complex patterns.

We use a `Pipeline` to ensure that `StandardScaler` is fitted only on the training data and then applied to the validation/test sets, preventing **data leakage**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42, early_stopping=True)
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

print("\nAll models trained successfully.")

## 9. Model Evaluation

We evaluate each model using a comprehensive set of metrics. Given the class imbalance identified in the EDA, **Macro** and **Weighted** averages are critical to understand how the models perform across all classes, especially the minority ones.

In [ ]:
def evaluate_models(pipelines, X, y):
    results = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y, y_pred),
            "Macro Precision": precision_score(y, y_pred, average='macro'),
            "Macro Recall": recall_score(y, y_pred, average='macro'),
            "Macro F1": f1_score(y, y_pred, average='macro'),
            "Weighted Precision": precision_score(y, y_pred, average='weighted'),
            "Weighted Recall": recall_score(y, y_pred, average='weighted'),
            "Weighted F1": f1_score(y, y_pred, average='weighted')
        }
        results.append(metrics)

    return pd.DataFrame(results)

# Evaluate on Validation Set
val_results = evaluate_models(trained_pipelines, X_val, y_val)
display(val_results.sort_values(by='Macro F1', ascending=False))

## 8. Machine Learning Models

We will train three different classification models to compare their performance:
1.  **Logistic Regression**: A baseline linear model for classification.
2.  **Random Forest**: An ensemble method that is robust to outliers and non-linear relationships.
3.  **Multilayer Perceptron (MLP)**: A neural network approach capable of learning complex patterns.

We use a `Pipeline` to ensure that `StandardScaler` is fitted only on the training data and then applied to the validation/test sets, preventing **data leakage**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42, early_stopping=True)
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

print("\nAll models trained successfully.")

## 9. Model Evaluation

We evaluate each model using a comprehensive set of metrics. Given the class imbalance identified in the EDA, **Macro** and **Weighted** averages are critical to understand how the models perform across all classes, especially the minority ones.

In [ ]:
def evaluate_models(pipelines, X, y):
    results = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y, y_pred),
            "Macro Precision": precision_score(y, y_pred, average='macro'),
            "Macro Recall": recall_score(y, y_pred, average='macro'),
            "Macro F1": f1_score(y, y_pred, average='macro'),
            "Weighted Precision": precision_score(y, y_pred, average='weighted'),
            "Weighted Recall": recall_score(y, y_pred, average='weighted'),
            "Weighted F1": f1_score(y, y_pred, average='weighted')
        }
        results.append(metrics)

    return pd.DataFrame(results)

# Evaluate on Validation Set
val_results = evaluate_models(trained_pipelines, X_val, y_val)
display(val_results.sort_values(by='Macro F1', ascending=False))

## 8. Machine Learning Models

We will train three different classification models to compare their performance:
1.  **Logistic Regression**: A baseline linear model for classification.
2.  **Random Forest**: An ensemble method that is robust to outliers and non-linear relationships.
3.  **Multilayer Perceptron (MLP)**: A neural network approach capable of learning complex patterns.

We use a `Pipeline` to ensure that `StandardScaler` is fitted only on the training data and then applied to the validation/test sets, preventing **data leakage**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42, early_stopping=True)
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

print("\nAll models trained successfully.")

## 9. Model Evaluation

We evaluate each model using a comprehensive set of metrics. Given the class imbalance identified in the EDA, **Macro** and **Weighted** averages are critical to understand how the models perform across all classes, especially the minority ones.

In [ ]:
def evaluate_models(pipelines, X, y):
    results = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y, y_pred),
            "Macro Precision": precision_score(y, y_pred, average='macro'),
            "Macro Recall": recall_score(y, y_pred, average='macro'),
            "Macro F1": f1_score(y, y_pred, average='macro'),
            "Weighted Precision": precision_score(y, y_pred, average='weighted'),
            "Weighted Recall": recall_score(y, y_pred, average='weighted'),
            "Weighted F1": f1_score(y, y_pred, average='weighted')
        }
        results.append(metrics)

    return pd.DataFrame(results)

# Evaluate on Validation Set
val_results = evaluate_models(trained_pipelines, X_val, y_val)
display(val_results.sort_values(by='Macro F1', ascending=False))

## 8. Machine Learning Models

We will train three different classification models to compare their performance:
1.  **Logistic Regression**: A baseline linear model for classification.
2.  **Random Forest**: An ensemble method that is robust to outliers and non-linear relationships.
3.  **Multilayer Perceptron (MLP)**: A neural network approach capable of learning complex patterns.

We use a `Pipeline` to ensure that `StandardScaler` is fitted only on the training data and then applied to the validation/test sets, preventing **data leakage**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42, early_stopping=True)
}

trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

print("\nAll models trained successfully.")

## 9. Model Evaluation

We evaluate each model using a comprehensive set of metrics. Given the class imbalance identified in the EDA, **Macro** and **Weighted** averages are critical to understand how the models perform across all classes, especially the minority ones.

In [ ]:
def evaluate_models(pipelines, X, y, dataset_name="Validation"):
    results = []
    for name, pipe in pipelines.items():
        y_pred = pipe.predict(X)

        metrics = {
            "Model": name,
            "Accuracy": accuracy_score(y, y_pred),
            "Macro Precision": precision_score(y, y_pred, average='macro'),
            "Macro Recall": recall_score(y, y_pred, average='macro'),
            "Macro F1": f1_score(y, y_pred, average='macro'),
            "Weighted Precision": precision_score(y, y_pred, average='weighted'),
            "Weighted Recall": recall_score(y, y_pred, average='weighted'),
            "Weighted F1": f1_score(y, y_pred, average='weighted')
        }
        results.append(metrics)

    return pd.DataFrame(results)

# Evaluate on Validation Set
val_results = evaluate_models(trained_pipelines, X_val, y_val, "Validation")
display(val_results.sort_values(by='Macro F1', ascending=False))